In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
import shap
import matplotlib.pyplot as plt
import os
import xgboost as xgb
import joblib

In [ ]:
dpath = r"D:\UKB\Clinical Biochemistry"
outfile = os.path.join(dpath, "XGB_feature_importance.csv")
model_path = os.path.join(dpath, "best_xgb_model.pkl")  # 模型保存路径
train_val_data = pd.read_csv(r"D:\Rdata and workplace\课题\4.16生化内部.csv")
external_data = pd.read_csv(r"D:\Rdata and workplace\课题\4.16生化外部.csv")

X_train = train_val_data.drop(columns=['status'])
y_train = train_val_data['status']

X_external = external_data.drop(columns=['status'])
y_external = external_data['status']
df_group = y_train
df_feature = X_train

In [ ]:
df_feature

In [ ]:
scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'learning_rate': 0.01,
    'max_depth': 3,
    'n_estimators': 400,
    'subsample': 0.8,
    'min_child_weight': 1,
    'gamma': 0.3,
    'colsample_bytree': 1.0,
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42,
    'n_jobs': 4
}

# 存储最佳模型信息
best_model = None
best_auc = 0
best_fold = 0
cv_metrics = []


for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    print(f"\n======= Fold {fold+1} =======")
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # 训练模型
    model = xgb.XGBClassifier(**params)
    model.fit(X_tr, y_tr, 
              eval_set=[(X_val, y_val)],
              verbose=0)
    
    # 验证集预测
    y_proba = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, y_proba)
    cv_metrics.append(fold_auc)
    print(f"Fold {fold+1} AUC: {fold_auc:.4f}")
    
    # 更新最佳模型
    if fold_auc > best_auc:
        best_auc = fold_auc
        best_model = model
        best_fold = fold + 1
        print(f"New best model found at Fold {best_fold} with AUC: {best_auc:.4f}")

# 保存最佳模型
joblib.dump(best_model, model_path)
print(f"\nBest model saved from Fold {best_fold} with AUC: {best_auc:.4f}")

# 交叉验证结果
print(f"\n=== Cross-validation Results ===")
print(f"Mean AUC: {np.mean(cv_metrics):.4f} (±{np.std(cv_metrics):.4f})")

# 使用最佳模型进行外部验证
print("\n=== External Validation with Best Model ===")
y_ext_proba = best_model.predict_proba(X_external)[:, 1]
ext_auc = roc_auc_score(y_external, y_ext_proba)
print(f"External Validation AUC: {ext_auc:.4f}")

# 1. 准备最佳折的内部验证集数据
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_count = 0

for train_idx, val_idx in cv.split(X_train, y_train):
    fold_count += 1
    if fold_count == best_fold:
        X_val_best = X_train.iloc[val_idx]
        y_val_best = y_train.iloc[val_idx]
        break

# 2. 预测概率
y_val_proba = best_model.predict_proba(X_val_best)[:, 1]
y_ext_proba = best_model.predict_proba(X_external)[:, 1]

# 3. 创建画布
plt.figure(figsize=(14, 6))



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb
from joblib import dump
from sklearn.metrics import roc_auc_score  # 添加这一行
# 初始化参数
scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'learning_rate': 0.01,
    'max_depth': 3,
    'n_estimators': 400,
    'subsample': 0.8,
    'min_child_weight': 1,
    'gamma': 0.3,
    'colsample_bytree': 1.0,
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42,
    'n_jobs': 49
}

# 存储数据容器
best_model_info = {
    'model': None,
    'fold': 0,
    'auc': 0,
    'val_idx': None,  # 存储验证集索引
    'fpr': None,
    'tpr': None
}
cv_metrics = []
all_fpr = []
all_tpr = []

# ================= 交叉验证流程 =================
for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    print(f"\n======= Fold {fold+1} =======")
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # 模型训练
    model = xgb.XGBClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=0)
    
    # 计算指标
    y_proba = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, y_proba)
    
    # 存储结果
    cv_metrics.append(fold_auc)
    fpr, tpr, _ = roc_curve(y_val, y_proba)
    all_fpr.append(fpr)
    all_tpr.append(tpr)
    
    # 更新最佳模型
    if fold_auc > best_model_info['auc']:
        best_model_info.update({
            'model': model,
            'fold': fold+1,
            'auc': fold_auc,
            'val_idx': val_idx,  # 保存验证集索引
            'fpr': fpr,
            'tpr': tpr
        })
        print(f"New best model at Fold {fold+1}, AUC = {fold_auc:.4f}")

# 保存最佳模型
dump(best_model_info['model'], 'best_model.joblib')
print(f"\nBest model from Fold {best_model_info['fold']}, AUC = {best_model_info['auc']:.4f}")

# ================= ROC曲线可视化 =================

###################
plt.figure(figsize=(6, 6))
plt.figure(figsize=(6, 6), facecolor='white')
ax = plt.gca()
ax.set_frame_on(True)  # 添加边框
ax.patch.set_edgecolor('black')  # 边框颜色为黑色
ax.patch.set_linewidth(1.5)  # 边框线宽
plt.figure(figsize=(6, 6), facecolor='white')
ax = plt.gca()
ax.set_frame_on(True)  # 添加边框
ax.patch.set_facecolor('white')  # 明确设置背景为白色
ax.patch.set_edgecolor('black')  # 边框颜色为黑色
ax.patch.set_linewidth(1.5)  # 边框线宽
plt.figure(figsize=(6, 6))


# 计算平均ROC曲线
mean_fpr = np.linspace(0, 1, 100)
mean_tpr = np.zeros_like(mean_fpr)
for i in range(5):
    mean_tpr += np.interp(mean_fpr, all_fpr[i], all_tpr[i])
mean_tpr /= 5
mean_auc = auc(mean_fpr, mean_tpr)


plt.plot(best_model_info['fpr'], best_model_info['tpr'], color='b', lw=2,
         label=f'Internal ( AUC = {best_model_info["auc"]:.2f})')

# 绘制外部验证曲线
y_ext_proba = best_model_info['model'].predict_proba(X_external)[:, 1]
fpr_ext, tpr_ext, _ = roc_curve(y_external, y_ext_proba)
roc_auc_ext = auc(fpr_ext, tpr_ext)
plt.plot(fpr_ext, tpr_ext, color='g', lw=2,
         label=f'External (AUC = {roc_auc_ext:.2f})')

# 格式设置
plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
plt.xlim([-0.01, 1.01])
plt.ylim([-0.01, 1.01])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Clinical Biochemistry ROC Curves', fontsize=14)
plt.legend(loc='lower right', frameon=True, facecolor='white')
plt.grid(alpha=0.3)
plt.tight_layout()
# 指定保存路径
save_dir = r"C:\Users\lenovo\Desktop\Figure"
os.makedirs(save_dir, exist_ok=True)  # 如果文件夹不存在，则创建
# ================= 混淆矩阵可视化 =================
# 获取最佳折数据
X_val_best = X_train.iloc[best_model_info['val_idx']]
y_val_best = y_train.iloc[best_model_info['val_idx']]

# 生成预测结果
y_val_pred = best_model_info['model'].predict(X_val_best)
y_ext_pred = best_model_info['model'].predict(X_external)

# 创建画布
plt.figure(figsize=(12, 5), facecolor='white')
ax1 = plt.subplot(1, 2, 1)
ax2 = plt.subplot(1, 2, 2)

# 内部验证矩阵
cm_val = confusion_matrix(y_val_best, y_val_pred)
disp_val = ConfusionMatrixDisplay(cm_val, display_labels=['Healthy', 'Stroke'])
disp_val.plot(cmap='Blues', ax=ax1, colorbar=False)
plt.title(f'Best Fold ({best_model_info["fold"]})\nAUC = {best_model_info["auc"]:.2f}')

# 添加数值标签
for i in range(2):
    for j in range(2):
        ax1.text(j, i, f"{cm_val[i, j]}",
                 ha="center", va="center",
                 color="white" if cm_val[i, j] > cm_val.max()/2 else "black")

# 外部验证矩阵
cm_ext = confusion_matrix(y_external, y_ext_pred)
disp_ext = ConfusionMatrixDisplay(cm_ext, display_labels=['Healthy', 'Stroke'])
disp_ext.plot(cmap='Oranges', ax=ax2, colorbar=False)
plt.title(f'External Validation\nAUC = {roc_auc_ext:.2f}')

# 添加数值标签
for i in range(2):
    for j in range(2):
        ax2.text(j, i, f"{cm_ext[i, j]}",
                 ha="center", va="center",
                 color="white" if cm_ext[i, j] > cm_ext.max()/2 else "black")

# 设置边框颜色和宽度
for ax in [ax1, ax2]:
    ax.set_frame_on(True)  # 添加边框
    ax.patch.set_edgecolor('black')  # 边框颜色为黑色
    ax.patch.set_linewidth(1.5)  # 边框线宽

plt.tight_layout()

# 保存为PDF
confusion_matrix_save_path = os.path.join(save_dir, "生化confusion_matrix.pdf")
plt.savefig(confusion_matrix_save_path, format='pdf', bbox_inches='tight', facecolor='white', dpi=300)
plt.show()  # 显示图形（可选）
# 保存为PDF
roc_save_path = os.path.join(save_dir, "生化roc_curve.pdf")
plt.savefig(roc_save_path, format='pdf', bbox_inches='tight', facecolor='white', dpi=300)
plt.show()  # 显示图形（可选）


# ================= 混淆矩阵可视化 =================
# 获取最佳折数据
X_val_best = X_train.iloc[best_model_info['val_idx']]
y_val_best = y_train.iloc[best_model_info['val_idx']]

# 生成预测结果
y_val_pred = best_model_info['model'].predict(X_val_best)
y_ext_pred = best_model_info['model'].predict(X_external)

# 创建画布
plt.figure(figsize=(12, 5))

# 内部验证矩阵
plt.subplot(1, 2, 1)
cm_val = confusion_matrix(y_val_best, y_val_pred)
disp_val = ConfusionMatrixDisplay(cm_val, display_labels=['Healthy', 'Stroke'])
disp_val.plot(cmap='Blues', ax=plt.gca(), colorbar=False)
plt.title(f'Best Fold ({best_model_info["fold"]})\nAUC = {best_model_info["auc"]:.2f}')

# 添加数值标签
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm_val[i, j]}",
                 ha="center", va="center",
                 color="white" if cm_val[i, j] > cm_val.max()/2 else "black")

# 外部验证矩阵
plt.subplot(1, 2, 2)
cm_ext = confusion_matrix(y_external, y_ext_pred)
disp_ext = ConfusionMatrixDisplay(cm_ext, display_labels=['Healthy', 'Stroke'])
disp_ext.plot(cmap='Oranges', ax=plt.gca(), colorbar=False)
plt.title(f'External Validation\nAUC = {roc_auc_ext:.2f}')

# 添加数值标签
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm_ext[i, j]}",
                 ha="center", va="center",
                 color="white" if cm_ext[i, j] > cm_ext.max()/2 else "black")

plt.savefig('生化confusion_matrix.pdf', format='pdf', bbox_inches='tight', facecolor='white', dpi=300)
plt.show()  # 显示图形（可选）

# ================= 性能报告 =================
print("\n" + "="*55)
print(f"{' Internal Validation Report ':=^55}")
print(classification_report(y_val_best, y_val_pred, target_names=['Healthy', 'Stroke']))

print("\n" + "="*55)
print(f"{' External Validation Report ':=^55}")
print(classification_report(y_external, y_ext_pred, target_names=['Healthy', 'Stroke']))
from sklearn.metrics import (roc_auc_score, accuracy_score, recall_score, 
                            precision_score, f1_score, matthews_corrcoef, 
                            confusion_matrix)

# ================= 性能指标表格输出 =================
from sklearn.metrics import (roc_auc_score, accuracy_score, recall_score, 
                            precision_score, f1_score, matthews_corrcoef, 
                            confusion_matrix)
import pandas as pd

def calculate_metrics(y_true, y_pred, y_proba):
    """计算全面的性能指标"""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    metrics = {
        'AUC': roc_auc_score(y_true, y_proba),
        'Accuracy': accuracy_score(y_true, y_pred),
        'Sensitivity': recall_score(y_true, y_pred),  # 敏感度/召回率
        'Specificity': tn / (tn + fp),               # 特异度
        'Youden Index': (recall_score(y_true, y_pred) + (tn / (tn + fp))) - 1,
        'PPV': precision_score(y_true, y_pred),      # 阳性预测值
        'NPV': tn / (tn + fn),                       # 阴性预测值
        'F1 Score': f1_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred)
    }
    return {k: round(v, 4) for k, v in metrics.items()}  # 保留4位小数

# 内部验证指标
y_val_best = y_train.iloc[best_model_info['val_idx']]
X_val_best = X_train.iloc[best_model_info['val_idx']]
y_val_pred = best_model_info['model'].predict(X_val_best)
y_val_proba = best_model_info['model'].predict_proba(X_val_best)[:, 1]
internal_metrics = calculate_metrics(y_val_best, y_val_pred, y_val_proba)

# 外部验证指标
y_ext_pred = best_model_info['model'].predict(X_external)
y_ext_proba = best_model_info['model'].predict_proba(X_external)[:, 1]
external_metrics = calculate_metrics(y_external, y_ext_pred, y_ext_proba)

# 创建并展示指标表格
metrics_df = pd.DataFrame({
    'Model': ['XGBoost (Internal)', 'XGBoost (External)'],
    'AUC': [internal_metrics['AUC'], external_metrics['AUC']],
    'Accuracy': [internal_metrics['Accuracy'], external_metrics['Accuracy']],
    'Sensitivity': [internal_metrics['Sensitivity'], external_metrics['Sensitivity']],
    'Specificity': [internal_metrics['Specificity'], external_metrics['Specificity']],
    'Youden Index': [internal_metrics['Youden Index'], external_metrics['Youden Index']],
    'PPV': [internal_metrics['PPV'], external_metrics['PPV']],
    'NPV': [internal_metrics['NPV'], external_metrics['NPV']],
    'F1 Score': [internal_metrics['F1 Score'], external_metrics['F1 Score']],
    'MCC': [internal_metrics['MCC'], external_metrics['MCC']]
})

# 打印表格
print("\n模型性能指标对比:")
print("="*100)
print(metrics_df.to_markdown(index=False))  # 使用markdown格式输出
print("="*100)

# 保存到CSV文件
metrics_df.to_csv('clinical_biochemistry_performance_metrics.csv', index=False)
print("\n指标已保存到 'clinical_biochemistry_performance_metrics.csv'")

# 额外打印详细指标（可选）
print("\n详细指标:")
print("\n内部验证:")
for metric, value in internal_metrics.items():
    print(f"{metric:>15}: {value:.4f}")

print("\n外部验证:")
for metric, value in external_metrics.items():
    print(f"{metric:>15}: {value:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import rcParams
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay

# ================= Global Formatting =================
rcParams['font.family'] = 'Arial'
rcParams['font.weight'] = 'normal'
plt.rc('axes', titlesize=14, labelsize=12)
plt.rc('legend', fontsize=11)

# ================= ROC Curve =================
plt.figure(figsize=(6, 6), facecolor='white')
ax = plt.gca()
ax.set_facecolor('white')

# Plot ROC curves
plt.plot(best_model_info['fpr'], best_model_info['tpr'], 
         color='#1f77b4', lw=2.5,  # Thicker line
         label=f'Internal (AUC = {best_model_info["auc"]:.2f})')

y_ext_proba = best_model_info['model'].predict_proba(X_external)[:, 1]
fpr_ext, tpr_ext, _ = roc_curve(y_external, y_ext_proba)
roc_auc_ext = auc(fpr_ext, tpr_ext)
plt.plot(fpr_ext, tpr_ext, 
         color='#ff7f0e', lw=2.5,  # Thicker line
         label=f'External (AUC = {roc_auc_ext:.2f})')

# Formatting
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.6)  # Darker reference line
plt.xlim([-0.01, 1.01])
plt.ylim([-0.01, 1.01])
plt.xlabel('False Positive Rate', fontsize=14, labelpad=8)  # Larger font
plt.ylabel('True Positive Rate', fontsize=14, labelpad=8)  # Larger font
plt.title('Biochemistry', fontsize=16, pad=12)  # More descriptive title
plt.legend(loc='lower right', frameon=True, 
           facecolor='white', edgecolor='black',
           framealpha=1, fontsize=12)  # Larger legend
plt.grid(alpha=0.2)  # Lighter grid

# Customize axes and ticks - keep only left and bottom spines with ticks
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(True)
ax.spines['bottom'].set_visible(True)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_color('black')

# Add tick marks
ax.tick_params(axis='both', which='both', length=5, width=1.5, color='black')
# 添加刻度线
ax.tick_params(axis='both', which='major',
                  length=6, width=1, color='black',
                  bottom=True, left=True)
plt.tight_layout()
plt.savefig(r"C:\Users\lenovo\Desktop\Figure\生化Roc_curve.pdf", 
            format='pdf', dpi=600,  # Higher resolution
            bbox_inches='tight', 
            facecolor='white', edgecolor='none')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import rcParams
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay

# ================= Global Formatting =================
rcParams['font.family'] = 'Arial'
rcParams['font.weight'] = 'normal'
plt.rc('axes', titlesize=14, labelsize=12)
plt.rc('legend', fontsize=14)  # 增大图例字体大小

# ================= ROC Curve =================
plt.figure(figsize=(6, 6), facecolor='white')
ax = plt.gca()
ax.set_facecolor('white')

# Plot ROC curves
plt.plot(best_model_info['fpr'], best_model_info['tpr'], 
         color='#1f77b4', lw=2.5,  # Thicker line
         label=f'Internal (AUC = {best_model_info["auc"]:.2f})')

y_ext_proba = best_model_info['model'].predict_proba(X_external)[:, 1]
fpr_ext, tpr_ext, _ = roc_curve(y_external, y_ext_proba)
roc_auc_ext = auc(fpr_ext, tpr_ext)
plt.plot(fpr_ext, tpr_ext, 
         color='#ff7f0e', lw=2.5,  # Thicker line
         label=f'External (AUC = {roc_auc_ext:.2f})')

# Formatting
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.6)  # Darker reference line
plt.xlim([-0.01, 1.01])
plt.ylim([-0.01, 1.01])
plt.xlabel('False positive rate', fontsize=14, labelpad=8)  # Larger font
plt.ylabel('True positive rate', fontsize=14, labelpad=8)  # Larger font
plt.title('Biochemistry', fontsize=16, pad=12)  # More descriptive title

# 增大图例并优化样式
legend = plt.legend(loc='lower right', frameon=True, 
           facecolor='white', edgecolor='black',
           framealpha=1, fontsize=14,  # 增大图例字体
           borderpad=1,  # 增加内边距
           handlelength=2,  # 增加图例句柄长度
           handletextpad=0.5)  # 增加文本与句柄间距
legend.get_frame().set_linewidth(1.5)  # 增加图例边框宽度

plt.grid(alpha=0.2)  # Lighter grid

# Customize axes and ticks - keep only left and bottom spines with ticks
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(True)
ax.spines['bottom'].set_visible(True)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_color('black')

# Add tick marks
ax.tick_params(axis='both', which='both', length=5, width=1.5, color='black')
# 添加刻度线
ax.tick_params(axis='both', which='major',
                  length=6, width=1, color='black',
                  bottom=True, left=True)
plt.tight_layout()
plt.tight_layout()
plt.savefig(r"C:\Users\lenovo\Desktop\Figure\生化Roc_curve.pdf", 
            format='pdf', dpi=600,  # Higher resolution
            bbox_inches='tight', 
            facecolor='white', edgecolor='none')
plt.show()

In [ ]:
# ================= ROC曲线可视化 =================
# 指定保存路径
save_dir = r"C:\Users\lenovo\Desktop\Figure"
os.makedirs(save_dir, exist_ok=True)  # 如果文件夹不存在，则创建
import matplotlib.pyplot as plt
from matplotlib import rcParams
import os

# Set global Arial font (SCI standard)
rcParams['font.family'] = 'Arial'
rcParams['font.weight'] = 'normal'

# 创建图形，设置背景为白色，添加黑色边框
plt.figure(figsize=(6, 6), facecolor='white')
ax = plt.gca()
ax.set_frame_on(True)  # 添加边框
ax.patch.set_facecolor('white')  # 明确设置背景为白色
ax.patch.set_edgecolor('black')  # 边框颜色为黑色
ax.patch.set_linewidth(1.5)  # 边框线宽

# 绘制最佳折的ROC曲线
plt.plot(best_model_info['fpr'], best_model_info['tpr'], color='#1f77b4', lw=2,
         label=f'Internal (AUC = {best_model_info["auc"]:.2f})')

# 绘制外部验证曲线
y_ext_proba = best_model_info['model'].predict_proba(X_external)[:, 1]
fpr_ext, tpr_ext, _ = roc_curve(y_external, y_ext_proba)
roc_auc_ext = auc(fpr_ext, tpr_ext)
plt.plot(fpr_ext, tpr_ext, color='#ff7f0e', lw=2,
         label=f'External (AUC = {roc_auc_ext:.2f})')

# 格式设置
plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
plt.xlim([-0.01, 1.01])
plt.ylim([-0.01, 1.01])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Biochemistry', fontsize=14)
plt.legend(loc='lower right', frameon=True, facecolor='white', edgecolor='black')  # 图例添加黑色边框
plt.grid(alpha=0.3)
plt.tight_layout()

# 确保整个图形的边框可见
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor('black')
    spine.set_linewidth(1.5)

# 保存为PDF
roc_save_path = os.path.join(save_dir, "生化roc_curve.pdf")
plt.savefig(roc_save_path, format='pdf', bbox_inches='tight', facecolor='white', dpi=300)
plt.show()  # 显示图形（可选）

In [ ]:
# ================= ROC曲线可视化 =================
import os
import matplotlib.pyplot as plt
from matplotlib import rcParams
from sklearn.metrics import roc_curve, auc

# 指定保存路径
save_dir = r"C:\Users\lenovo\Desktop\Figure"
os.makedirs(save_dir, exist_ok=True)  # 如果文件夹不存在，则创建

# 设置全局Arial字体 (SCI标准)
rcParams['font.family'] = 'Arial'
rcParams['font.weight'] = 'normal'
rcParams['font.size'] = 12  # 设置全局字体大小

# 创建图形，设置背景为白色，添加黑色边框
plt.figure(figsize=(6, 6), facecolor='white')
ax = plt.gca()
ax.set_frame_on(True)  # 添加边框
ax.patch.set_facecolor('white')  # 明确设置背景为白色
ax.patch.set_edgecolor('black')  # 边框颜色为黑色
ax.patch.set_linewidth(1.5)  # 边框线宽

# 绘制内部验证的ROC曲线
plt.plot(best_model_info['fpr'], best_model_info['tpr'], color='#1f77b4', lw=2,
         label=f'Internal (AUC = {best_model_info["auc"]:.2f})')

# 绘制外部验证的ROC曲线
y_ext_proba = best_model_info['model'].predict_proba(X_external)[:, 1]
fpr_ext, tpr_ext, _ = roc_curve(y_external, y_ext_proba)
roc_auc_ext = auc(fpr_ext, tpr_ext)
plt.plot(fpr_ext, tpr_ext, color='#ff7f0e', lw=2,
         label=f'External (AUC = {roc_auc_ext:.2f})')

# 格式设置
plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)  # 对角线
plt.xlim([-0.01, 1.01])
plt.ylim([-0.01, 1.01])
plt.xlabel('False Positive Rate', fontsize=14)  # 增大字体大小
plt.ylabel('True Positive Rate', fontsize=14)   # 增大字体大小
plt.title('Biochemistry', fontsize=16)          # 增大标题字体大小
plt.legend(loc='lower right', frameon=True, facecolor='white', edgecolor='black')  # 图例添加黑色边框
plt.grid(alpha=0.3)
plt.tight_layout()

# 确保整个图形的边框可见
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor('black')
    spine.set_linewidth(1.5)

# 保存为PDF
roc_save_path = os.path.join(save_dir, "生化roc_curve.pdf")
plt.savefig(roc_save_path, format='pdf', bbox_inches='tight', facecolor='white', dpi=300)
plt.show()  # 显示图形（可选）


In [ ]:
# ================= ROC Curve Visualization =================
import matplotlib.pyplot as plt
from matplotlib import rcParams
import os

# Set global Arial font (SCI standard)
rcParams['font.family'] = 'Arial'
rcParams['font.weight'] = 'normal'

# Create save directory
save_dir = r"C:\Users\lenovo\Desktop\Figure"
os.makedirs(save_dir, exist_ok=True)

# Create figure with white background and black border
plt.figure(figsize=(6, 6), facecolor='white', dpi=600)  # High resolution
ax = plt.gca()
ax.set_frame_on(True)
ax.patch.set_facecolor('white')
ax.patch.set_edgecolor('black')
ax.patch.set_linewidth(1.5)

# Plot ROC curves with thicker lines
plt.plot(best_model_info['fpr'], best_model_info['tpr'], 
         color='#1f77b4', lw=2.5,  # Thicker line
         label=f'Internal (AUC = {best_model_info["auc"]:.2f})')

# External validation curve
y_ext_proba = best_model_info['model'].predict_proba(X_external)[:, 1]
fpr_ext, tpr_ext, _ = roc_curve(y_external, y_ext_proba)
roc_auc_ext = auc(fpr_ext, tpr_ext)
plt.plot(fpr_ext, tpr_ext, 
         color='#ff7f0e', lw=2.5,  # Thicker line
         label=f'External (AUC = {roc_auc_ext:.2f})')

# Formatting
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.6)  # Darker reference line
plt.xlim([-0.01, 1.01])
plt.ylim([-0.01, 1.01])

# Enhanced labels with larger font
plt.xlabel('False Positive Rate', fontsize=14, labelpad=8)
plt.ylabel('True Positive Rate', fontsize=14, labelpad=8)
plt.title('Biochemistry', fontsize=16, pad=12)

# Improved legend
plt.legend(loc='lower right', frameon=True,
           facecolor='white', edgecolor='black',
           fontsize=12, framealpha=1)  # Larger font size

plt.grid(alpha=0.2)  # Lighter grid

# Border styling
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor('black')
    spine.set_linewidth(1.5)

plt.tight_layout()

# Save with publication quality
roc_save_path = os.path.join(save_dir, "Clinical_Biochemistry_ROC.pdf")
plt.savefig(roc_save_path, 
            format='pdf', 
            dpi=600,
            bbox_inches='tight',
            facecolor='white',
            edgecolor='black')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, accuracy_score, recall_score, 
                            precision_score, f1_score, matthews_corrcoef, 
                            confusion_matrix)
from scipy import stats
import xgboost as xgb
import joblib

# 函数：计算指标及其95%置信区间
def calculate_metrics_with_ci(y_true, y_pred, y_proba, n_bootstrap=1000):
    """计算分类指标及95%置信区间(使用bootstrap方法)"""
    metrics_dict = {
        'AUC': roc_auc_score,
        'Accuracy': accuracy_score,
        'Sensitivity': recall_score,
        'PPV': precision_score,
        'F1 Score': f1_score,
        'MCC': matthews_corrcoef
    }
    
    # 初始化结果存储
    results = {}
    bootstrap_results = {metric: [] for metric in metrics_dict}
    
    # 计算点估计
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    point_estimates = {
        'AUC': roc_auc_score(y_true, y_proba),
        'Accuracy': accuracy_score(y_true, y_pred),
        'Sensitivity': recall_score(y_true, y_pred),
        'Specificity': tn / (tn + fp),
        'Youden Index': (recall_score(y_true, y_pred) + (tn / (tn + fp))) - 1,
        'PPV': precision_score(y_true, y_pred),
        'NPV': tn / (tn + fn),
        'F1 Score': f1_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred)
    }
    
    # Bootstrap抽样计算置信区间
    np.random.seed(42)
    n_samples = len(y_true)
    for _ in range(n_bootstrap):
        indices = np.random.choice(n_samples, n_samples, replace=True)
        y_true_boot = y_true.iloc[indices]
        y_pred_boot = y_pred[indices]
        y_proba_boot = y_proba[indices]
        
        for metric, func in metrics_dict.items():
            try:
                if metric == 'AUC':
                    score = func(y_true_boot, y_proba_boot)
                else:
                    score = func(y_true_boot, y_pred_boot)
                bootstrap_results[metric].append(score)
            except:
                bootstrap_results[metric].append(np.nan)
    
    # 计算置信区间
    ci_results = {}
    for metric in metrics_dict:
        scores = bootstrap_results[metric]
        lower = np.percentile(scores, 2.5)
        upper = np.percentile(scores, 97.5)
        ci_results[metric] = (lower, upper)
    
    # 计算特异性、Youden Index和NPV的置信区间
    tn_list, fp_list, fn_list, tp_list = [], [], [], []
    for _ in range(n_bootstrap):
        indices = np.random.choice(n_samples, n_samples, replace=True)
        cm = confusion_matrix(y_true.iloc[indices], y_pred[indices]).ravel()
        if len(cm) == 4:
            tn, fp, fn, tp = cm
            tn_list.append(tn)
            fp_list.append(fp)
            fn_list.append(fn)
            tp_list.append(tp)
    
    specificity_ci = np.percentile(np.array(tn_list)/(np.array(tn_list)+np.array(fp_list)), [2.5, 97.5])
    npv_ci = np.percentile(np.array(tn_list)/(np.array(tn_list)+np.array(fn_list)), [2.5, 97.5])
    youden_ci = np.percentile(
        (np.array(tp_list)/(np.array(tp_list)+np.array(fn_list))) + 
        (np.array(tn_list)/(np.array(tn_list)+np.array(fp_list))) - 1,
        [2.5, 97.5]
    )
    
    ci_results.update({
        'Specificity': specificity_ci,
        'Youden Index': youden_ci,
        'NPV': npv_ci
    })
    
    # 格式化结果
    formatted_results = {}
    for metric in point_estimates:
        ci = ci_results.get(metric, (np.nan, np.nan))
        formatted_results[metric] = {
            'Value': point_estimates[metric],
            'CI_lower': ci[0],
            'CI_upper': ci[1],
            'Formatted': f"{point_estimates[metric]:.3f} ({ci[0]:.3f}-{ci[1]:.3f})"
        }
    
    return formatted_results

# 计算内部验证集指标
int_metrics = calculate_metrics_with_ci(y_val_best, y_val_pred, y_val_proba)

# 计算外部验证集指标
ext_metrics = calculate_metrics_with_ci(y_external, y_ext_pred, y_ext_proba)

# 创建结果DataFrame
metrics_list = ['AUC', 'Accuracy', 'Sensitivity', 'Specificity', 
                'Youden Index', 'PPV', 'NPV', 'F1 Score', 'MCC']

results_data = []
for metric in metrics_list:
    results_data.append({
        'Metric': metric,
        'Internal': int_metrics[metric]['Formatted'],
        'External': ext_metrics[metric]['Formatted'],
        'Internal_Value': int_metrics[metric]['Value'],
        'Internal_CI': (int_metrics[metric]['CI_lower'], int_metrics[metric]['CI_upper']),
        'External_Value': ext_metrics[metric]['Value'],
        'External_CI': (ext_metrics[metric]['CI_lower'], ext_metrics[metric]['CI_upper'])
    })

results_df = pd.DataFrame(results_data)

# 打印美观的表格
print("\n=== Model Performance Metrics with 95% Confidence Intervals ===")
print(results_df[['Metric', 'Internal', 'External']].to_markdown(index=False))

# 可视化部分指标对比
plt.figure(figsize=(12, 6))
metrics_to_plot = ['AUC', 'Accuracy', 'Sensitivity', 'Specificity', 'F1 Score']

for i, metric in enumerate(metrics_to_plot):
    plt.subplot(2, 3, i+1)
    
    # 内部验证结果
    int_val = results_df.loc[results_df['Metric'] == metric, 'Internal_Value'].values[0]
    int_ci = results_df.loc[results_df['Metric'] == metric, 'Internal_CI'].values[0]
    
    # 外部验证结果
    ext_val = results_df.loc[results_df['Metric'] == metric, 'External_Value'].values[0]
    ext_ci = results_df.loc[results_df['Metric'] == metric, 'External_CI'].values[0]
    
    plt.bar(['Internal', 'External'], [int_val, ext_val], 
            yerr=[[int_val - int_ci[0], ext_val - ext_ci[0]], 
                  [int_ci[1] - int_val, ext_ci[1] - ext_val]],
            capsize=5, color=['blue', 'orange'])
    plt.title(metric)
    plt.ylim(0, 1.05)
    if metric == 'AUC':
        plt.axhline(0.5, linestyle='--', color='gray', alpha=0.5)
    else:
        plt.axhline(0, linestyle='--', color='gray', alpha=0.5)

plt.tight_layout()
plt.suptitle('Model Performance Metrics Comparison with 95% CI', y=1.02)
plt.show()

# 保存详细结果到CSV
detailed_results = results_df.copy()
detailed_results['Internal_Value'] = detailed_results['Internal_Value'].round(4)
detailed_results['Internal_CI_lower'] = detailed_results['Internal_CI'].apply(lambda x: x[0].round(4))
detailed_results['Internal_CI_upper'] = detailed_results['Internal_CI'].apply(lambda x: x[1].round(4))
detailed_results['External_Value'] = detailed_results['External_Value'].round(4)
detailed_results['External_CI_lower'] = detailed_results['External_CI'].apply(lambda x: x[0].round(4))
detailed_results['External_CI_upper'] = detailed_results['External_CI'].apply(lambda x: x[1].round(4))

detailed_results.drop(['Internal_CI', 'External_CI'], axis=1).to_csv(
    'model_performance_with_ci.csv', index=False)

In [ ]:
# 计算特征的Spearman相关性矩阵
correlation_matrix = df_feature.corr(method='spearman')

# 打印相关性矩阵
print(correlation_matrix)

# 保存相关性矩阵为CSV文件
correlation_matrix.to_csv(dpath + 'Stroke_hc_correlation_matrix.csv', index=True)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster import hierarchy
from scipy.spatial.distance import pdist, squareform

pdf_output_file = r'D:\UKB\Clinical Biochemistry\Stroke_hc_correlation_matrix_heatmap.pdf'

# 将相关性矩阵转换为距离矩阵 (1 - 绝对值相关性)(相关性越强（|r|→1），距离越近（→0）)
distance_matrix = 1 - np.abs(correlation_matrix)

# 计算层次聚类的链式方法 (使用 Ward 方法)
# pdist 用于计算距离矩阵的压缩形式
dist_array = squareform(distance_matrix)  # 转换为压缩距离矩阵
dist_linkage = hierarchy.linkage(dist_array, method='ward')

In [ ]:
# 创建单个子图用于绘制树状图
fig, ax = plt.subplots(figsize=(30, 16))
# 计算层次聚类并生成树状图 (Dendrogram)
dendro = hierarchy.dendrogram(dist_linkage, labels=correlation_matrix.columns, ax=ax)
# 设置树状图的 x 轴标签
ax.set_xticklabels(dendro["ivl"], rotation=60, fontsize=4, horizontalalignment='right')
# 绘制水平线以标识聚类阈值 (可以根据需要调整y的值)
ax.axhline(y=0.5, color='r', linewidth=2, linestyle='--')
# 保存图像为文件 (PDF 或 PNG，或其他格式)
plt.savefig(pdf_output_file, dpi=300, format='pdf')  # 保存为PDF文件，确保图像的高分辨率
# 显示图像
plt.show()

In [ ]:
from collections import defaultdict
from scipy.cluster import hierarchy

# 聚类，距离阈值为0.5，按簇划分
cluster_ids = hierarchy.fcluster(dist_linkage, 0.5, criterion="distance")

# 创建字典，将簇ID映射到特征的索引(每个特征被分配到一个簇ID)
cluster_id_to_feature_ids = defaultdict(list)

# 将特征的索引按簇ID分类(同一簇内的特征高度相关（Spearman |r| > 0.5）)
for idx, cluster_id in enumerate(cluster_ids):
    cluster_id_to_feature_ids[cluster_id].append(idx)

# 输出聚类结果（每个簇的特征索引）
for cluster_id, feature_ids in cluster_id_to_feature_ids.items():
    print(f"Cluster {cluster_id}: Feature indices {feature_ids}")

# 将特征的簇ID与特征名称对应
feature_names = df_feature.columns.tolist()  # 获取特征名称
cluster_data = pd.DataFrame({
    'Feature': feature_names,
    'ClusterID': cluster_ids
})

# 输出文件路径
output_csv = r'D:\UKB\Clinical Biochemistry\Stroke_feature_clusters.csv'
# 将结果保存为CSV文件
cluster_data.to_csv(output_csv, index=False)
print(f"聚类结果已保存至: {output_csv}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import xgboost as xgb
from tqdm import tqdm  # 用于显示进度条

# 数据准备
dpath = "D:/UKB/Clinical Biochemistry/"
output_file = dpath + 'RNA_auc_scores.csv'

# 计算类别权重
scale_pos_weight = [np.sum(y_train == 0) / np.sum(y_train == 1)]  # 改为列表形式

# 设置模型参数 (全部使用列表形式)
params = {
    'learning_rate': [0.01],
    'max_depth': [3],
    'n_estimators': [400],
    'subsample': [0.7],
    'min_child_weight': [1],  # 从3改为1
    'gamma': [0.1],  # 从0.9改为0.1
    'colsample_bytree': [1.0],
    'random_state': [42],
    'scale_pos_weight': scale_pos_weight,
    'eval_metric': ['auc'],
    'objective': ['binary:logistic']
}

# 交叉验证设置 (改为5折)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def calculate_auc_for_feature(RNA_feature, df_feature, df_group, cv, params):
    """计算单个特征的交叉验证AUC"""
    aucs = []
    
    # 展开参数 (因为XGBoost不接受列表形式的单个参数)
    model_params = {k: v[0] for k, v in params.items()}
    
    for train_idx, test_idx in cv.split(df_feature, df_group):
        # 数据划分
        X_train = df_feature.iloc[train_idx][[RNA_feature]]
        X_test = df_feature.iloc[test_idx][[RNA_feature]]
        y_train = df_group.iloc[train_idx]
        y_test = df_group.iloc[test_idx]
        
        # 模型训练和预测
        model = xgb.XGBClassifier(**model_params)
        model.fit(X_train, y_train)
        y_pred_prob = model.predict_proba(X_test)[:, 1]
        aucs.append(roc_auc_score(y_test, y_pred_prob))
    
    return np.mean(aucs)

# 计算每个RNA的AUC (添加进度条)
auc_scores = {}
for RNA_feature in tqdm(df_feature.columns, desc="Evaluating features"):
    auc_score = calculate_auc_for_feature(RNA_feature, df_feature, df_group, cv, params)
    auc_scores[RNA_feature] = auc_score

# 创建结果DataFrame并排序
auc_df = pd.DataFrame(list(auc_scores.items()), columns=['Feature', 'AUC'])
auc_df = auc_df.sort_values('AUC', ascending=False)

# 保存结果
auc_df.to_csv(output_file, index=False)
print(f"\nAUC scores saved to: {output_file}")
print("\nTop 10 features by AUC:")
print(auc_df.head(10))

# 可视化Top特征
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.barh(auc_df['Feature'].head(20)[::-1], auc_df['AUC'].head(20)[::-1])
plt.xlabel('AUC Score')
plt.title('Top 20 Predictive Features (5-fold CV)')
plt.tight_layout()
plt.show()

In [ ]:
feature_auc = pd.read_csv('D:/UKB/Clinical Biochemistry/RNA_auc_scores.csv', encoding='GBK')
feature_clus = pd.read_csv(r"D:\UKB\Clinical Biochemistry\Stroke_feature_clusters.csv", encoding='GBK')

feature_sel = pd.merge(feature_auc, feature_clus, how='left', on="Feature")
best_features = feature_sel.loc[feature_sel.groupby('ClusterID')['AUC'].idxmax()]

# 输出选择后的特征
output_file = dpath + 'RNA_best_features.csv'
best_features.to_csv(output_file, index=False)

In [ ]:
feature482 = best_features['Feature']

In [ ]:
feature482

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, accuracy_score, confusion_matrix, 
                            roc_curve, precision_score, recall_score, f1_score, 
                            matthews_corrcoef, ConfusionMatrixDisplay)
import xgboost as xgb
import joblib
from tqdm import tqdm

# 1. 数据准备
feature482 = best_features['Feature']
X_train_selected = X_train[feature482]
X_external_selected = X_external[feature482]
X_train_selected

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, confusion_matrix, 
                            roc_curve, auc, ConfusionMatrixDisplay)
import xgboost as xgb

# 数据准备
feature482 = best_features['Feature']
X_train_selected = X_train[feature482]
X_external_selected = X_external[feature482]

# 计算类别权重
scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)

# 模型参数
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'learning_rate': 0.01,
    'max_depth': 3,
    'n_estimators': 400,
    'subsample': 0.7,
    'min_child_weight': 1,
    'gamma': 0.1,
    'colsample_bytree': 1.0,
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42,
    'n_jobs': 4
}

# 五折交叉验证
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_model = None
best_auc = 0
best_fold = 0

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_selected, y_train), 1):
    X_tr, X_val = X_train_selected.iloc[train_idx], X_train_selected.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_tr, y_tr)
    
    y_proba = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, y_proba)
    
    if fold_auc > best_auc:
        best_auc = fold_auc
        best_model = model
        best_fold = fold
        X_best_val = X_val
        y_best_val = y_val

# 内部验证可视化
plt.figure(figsize=(12, 5))

# 内部验证ROC
plt.subplot(1, 2, 1)
y_val_proba = best_model.predict_proba(X_best_val)[:, 1]
fpr, tpr, _ = roc_curve(y_best_val, y_val_proba)
plt.plot(fpr, tpr, label=f'Fold {best_fold} (AUC={best_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Internal Validation ROC')
plt.legend()

# 内部验证混淆矩阵
plt.subplot(1, 2, 2)
y_val_pred = best_model.predict(X_best_val)
cm = confusion_matrix(y_best_val, y_val_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['No Stroke', 'Stroke'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Internal Validation CM')

plt.tight_layout()
plt.show()

# 外部验证可视化
plt.figure(figsize=(12, 5))

# 外部验证ROC
plt.subplot(1, 2, 1)
y_ext_proba = best_model.predict_proba(X_external_selected)[:, 1]
fpr_ext, tpr_ext, _ = roc_curve(y_external, y_ext_proba)
roc_auc_ext = auc(fpr_ext, tpr_ext)
plt.plot(fpr_ext, tpr_ext, 'r', label=f'External (AUC={roc_auc_ext:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('External Validation ROC')
plt.legend()

# 外部验证混淆矩阵
plt.subplot(1, 2, 2)
y_ext_pred = best_model.predict(X_external_selected)
cm_ext = confusion_matrix(y_external, y_ext_pred)
disp_ext = ConfusionMatrixDisplay(cm_ext, display_labels=['No Stroke', 'Stroke'])
disp_ext.plot(cmap='Reds', values_format='d')
plt.title('External Validation CM')

plt.tight_layout()
plt.show()

In [ ]:
df_feature = df_feature[feature482]

df_feature.insert(0, 'status', train_val_data['status'])

df_feature.to_csv('D:/UKB/Clinical Biochemistry/rmRmMultiColin_feature482.csv', index=False)
df_feature = df_feature.drop(columns=['status'])

In [ ]:
df_feature

In [ ]:
import xgboost as xgb
from xgboost import XGBClassifier
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
import shap
from collections import Counter
# 计算类别权重
scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)
# 设置模型参数
params = {
    'n_estimators': 400,
    'learning_rate': 0.01,
    'max_depth': 3,
    'subsample': 0.7,
    'min_child_weight': 3,
    'gamma': 0.9,
    'eval_metric': 'auc',
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42
}
# 交叉验证设置
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 函数：标准化重要性
def normal_imp(mydict):
    mysum = sum(mydict.values())
    for key in mydict.keys():
        mydict[key] = mydict[key] / mysum
    return mydict

# 初始化重要性计数器
tg_imp_cv = Counter()
shap_imp_cv = np.zeros(df_feature.shape[1])  # 初始化SHAP重要性数组

# 交叉验证过程
for train_idx, test_idx in cv.split(df_feature, df_group):
    X_train, X_test = df_feature.iloc[train_idx, :], df_feature.iloc[test_idx, :]
    y_train, y_test = df_group.iloc[train_idx], df_group.iloc[test_idx]

    # 训练 XGB 分类器
    my_xgb = XGBClassifier(**params)
    my_xgb.fit(X_train, y_train)

    # 计算总增益重要性
    totalgain_imp = my_xgb.feature_importances_  # 直接使用 feature_importances_
    totalgain_imp = dict(zip(df_feature.columns, totalgain_imp.tolist()))

    # # 计算总覆盖率重要性，xgb没有办法计算
    # totalcover_imp = my_xgb.booster_.feature_importance(importance_type='split')
    # totalcover_imp = dict(zip(df_feature.columns, totalcover_imp.tolist()))

    # 更新重要性计数器
    tg_imp_cv += Counter(normal_imp(totalgain_imp))

    # 计算 SHAP 值
    explainer = shap.TreeExplainer(my_xgb)
    shap_values = explainer.shap_values(X_test)
    # 取绝对值的平均 SHAP 值，确保分母合理
    shap_values_mean = np.mean(np.abs(shap_values), axis=0)
    shap_imp_cv += shap_values_mean / np.sum(shap_values_mean)  # 归一化

In [ ]:
df_feature

In [ ]:
shap_values_mean.shape

In [ ]:
feature482

In [ ]:
# 创建 SHAP 重要性数据框
shap_imp_df = pd.DataFrame({
    'Analytes': df_feature.columns,
    'ShapValues_cv': shap_imp_cv / 10
})
shap_imp_df.sort_values(by='ShapValues_cv', ascending=False, inplace=True)

# 计算基本统计信息
stats_summary = {
    'Mean': shap_imp_df['ShapValues_cv'].mean(),
    'Std': shap_imp_df['ShapValues_cv'].std(),
    'Min': shap_imp_df['ShapValues_cv'].min(),
    'Max': shap_imp_df['ShapValues_cv'].max(),
    '25%': shap_imp_df['ShapValues_cv'].quantile(0.25),
    '50% (Median)': shap_imp_df['ShapValues_cv'].median(),
    '75%': shap_imp_df['ShapValues_cv'].quantile(0.75)
}

# 打印统计信息
print("SHAP Values CV Statistics:")
for stat, value in stats_summary.items():
    print(f"{stat}: {value:.4f}")

In [ ]:
# 创建总增益重要性数据框
tg_imp_cv = normal_imp(tg_imp_cv)
tg_imp_df = pd.DataFrame({
    'Analytes': list(tg_imp_cv.keys()),
    'TotalGain_cv': list(tg_imp_cv.values())
})
tg_imp_df

In [ ]:
outfile

In [ ]:
# 合并所有重要性数据框
my_imp_df = pd.merge(left=shap_imp_df, right=tg_imp_df, how='left', on=['Analytes'])

# 计算综合重要性
my_imp_df['Ensemble_cv'] = (my_imp_df['ShapValues_cv'] + my_imp_df['TotalGain_cv']) / 2
my_imp_df.sort_values(by='TotalGain_cv', ascending=False, inplace=True)

# 保存结果
my_imp_df.to_csv(outfile, index=False)

print('finished')

In [ ]:
# 筛选前90%增益特征，不只使用TotalGain_cv，使用Ensemble_cv
def get_imp_analy(my_imp_df, top_prop=0.9):
    imp_score, iter = 0, 0
    # 遍历 my_imp_df 的 Ensemble_cv 列，累加直到累计超过 top_prop
    while imp_score < top_prop and iter < len(my_imp_df):
        imp_score += my_imp_df.Ensemble_cv.iloc[iter]  # 使用 iloc 获取第 iter 行的值
        iter += 1
    return iter  # 返回累积达到 top_prop 时的行索引

# 调用函数，使用 Ensemble_cv
top_feature_count = get_imp_analy(my_imp_df, top_prop=0.9)

print(f"Number of proteins contributing to over 90% of overall information gains: {top_feature_count}")

# 获取前 top_feature_count 个特征
top_features = my_imp_df.iloc[:top_feature_count]

# 保存到 CSV 文件
dpath = 'D:/UKB/Clinical Biochemistry'
top_features.to_csv(dpath + 'Top_40_InfoGain_features.csv', index=False)

print('Top-ranked proteins saved to CSV.')

In [ ]:
# 获取前 top_feature_count 个特征
top_features = my_imp_df.iloc[:top_feature_count]

# 保存到 CSV 文件

top_features.to_csv(dpath + 'Top_40_InfoGain_features.csv', index=False)

print('Top-ranked proteins saved to CSV.')

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
import shap
from collections import Counter

dpath = 'D:/UKB/Clinical Biochemistry'
# 加载数据
top_features_df = pd.read_csv(dpath + 'Top_40_InfoGain_features.csv')
top_features = top_features_df['Analytes'].tolist()
top_features_df

In [ ]:
train_val_data = pd.read_csv(r"D:\Rdata and workplace\课题\4.16生化内部.csv")
external_data = pd.read_csv(r"D:\Rdata and workplace\课题\4.16生化外部.csv")

X_train = train_val_data.drop(columns=['status'])
y_train = train_val_data['status']

X_external = external_data.drop(columns=['status'])
y_external = external_data['status']
df_group = y_train
df_feature = X_train

In [ ]:
df_feature = X_train[top_features]
df_feature

In [ ]:
X_train = df_feature
X_train

In [ ]:
# 数据准备
X_train_selected = X_train[top_features]
X_external_selected = X_external[top_features]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
import xgboost as xgb

# 数据准备
X_train_selected = X_train[top_features]
X_external_selected = X_external[top_features]

# 计算类别权重
scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)

# 模型参数
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'learning_rate': 0.01,
    'max_depth': 3,
    'n_estimators': 400,
    'subsample': 0.7,
    'min_child_weight': 3,
    'gamma': 0.9,
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42,
    'n_jobs': 4
}

# 五折交叉验证
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_model = None
best_auc = 0
best_fold = 0

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_selected, y_train), 1):
    X_tr, X_val = X_train_selected.iloc[train_idx], X_train_selected.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_tr, y_tr)
    
    y_proba = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, y_proba)
    
    if fold_auc > best_auc:
        best_auc = fold_auc
        best_model = model
        best_fold = fold
        X_best_val = X_val
        y_best_val = y_val

# 内部验证可视化
plt.figure(figsize=(12, 5))

# 内部ROC曲线
plt.subplot(1, 2, 1)
y_val_proba = best_model.predict_proba(X_best_val)[:, 1]
fpr, tpr, _ = roc_curve(y_best_val, y_val_proba)
plt.plot(fpr, tpr, label=f'Fold {best_fold} (AUC={best_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Internal Validation ROC')
plt.legend()



In [ ]:
import numpy as np
import pandas as pd
import scipy.stats
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np
import scipy.stats
from scipy.stats import norm
from scipy import stats


# AUC comparison adapted from
# https://github.com/Netflix/vmaf/
def compute_midrank(x):
    """Computes midranks."""
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1)
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T + 1
    return T2


def fastDeLong(predictions_sorted_transposed, label_1_count):
    """
    The fast version of DeLong's method for computing the covariance of
    unadjusted AUC.
    Args:
       predictions_sorted_transposed: a 2D numpy.array[n_classifiers, n_examples]
          sorted such as the examples with label "1" are first
    Returns:
       (AUC value, DeLong covariance)
    Reference:
     @article{sun2014fast,
       title={Fast Implementation of DeLong's Algorithm for
              Comparing the Areas Under Correlated Receiver Operating Characteristic Curves},
       author={Xu Sun and Weichao Xu},
       journal={IEEE Signal Processing Letters},
       volume={21},
       number={11},
       pages={1389--1393},
       year={2014},
       publisher={IEEE}
     }
    """
    # Short variables are named as they are in the paper
    m = label_1_count
    n = predictions_sorted_transposed.shape[1] - m
    positive_examples = predictions_sorted_transposed[:, :m]
    negative_examples = predictions_sorted_transposed[:, m:]
    k = predictions_sorted_transposed.shape[0]

    tx = np.empty([k, m], dtype=float)
    ty = np.empty([k, n], dtype=float)
    tz = np.empty([k, m + n], dtype=float)
    for r in range(k):
        tx[r, :] = compute_midrank(positive_examples[r, :])
        ty[r, :] = compute_midrank(negative_examples[r, :])
        tz[r, :] = compute_midrank(predictions_sorted_transposed[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    sx = np.cov(v01)
    sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov


def calc_pvalue(aucs, covar):
    """Computes log(10) of p-values.
    Args:
       aucs: 1D array of AUCs
       covar: AUC DeLong covariances
    Returns:
       log10(pvalue)
    """
    l = np.array([[1, -1]])
    z = np.abs(np.diff(aucs)) / np.sqrt(np.dot(np.dot(l, covar), l.T))  # 将 sigma 改为 covar
    return np.log10(2) + scipy.stats.norm.logsf(z, loc=0, scale=1) / np.log(10)


def compute_ground_truth_statistics(ground_truth):
    assert np.array_equal(np.unique(ground_truth), [0, 1])
    order = (-ground_truth).argsort()
    label_1_count = int(ground_truth.sum())
    return order, label_1_count


def delong_roc_variance(ground_truth, predictions):
    """
    Computes ROC AUC variance for a single set of predictions
    Args:
       ground_truth: np.array of 0 and 1
       predictions: np.array of floats of the probability of being class 1
    """
    order, label_1_count = compute_ground_truth_statistics(ground_truth)
    predictions_sorted_transposed = predictions[np.newaxis, order]
    aucs, delongcov = fastDeLong(predictions_sorted_transposed, label_1_count)
    assert len(aucs) == 1, "There is a bug in the code, please forward this to the developers"
    return aucs[0], delongcov


def delong_roc_test(ground_truth, predictions_one, predictions_two):
    """
    Computes log(p-value) for hypothesis that two ROC AUCs are different
    Args:
       ground_truth: np.array of 0 and 1
       predictions_one: predictions of the first model,
          np.array of floats of the probability of being class 1
       predictions_two: predictions of the second model,
          np.array of floats of the probability of being class 1
    """
    order, label_1_count = compute_ground_truth_statistics(ground_truth)
    predictions_sorted_transposed = np.vstack((predictions_one, predictions_two))[:, order]
    aucs, delongcov = fastDeLong(predictions_sorted_transposed, label_1_count)
    return calc_pvalue(aucs, delongcov)



In [ ]:
# 计算类别权重
scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)
# 设置模型参数
params = {
   'n_estimators': 400,
    'learning_rate': 0.01,
    'max_depth': 3,
    'subsample': 0.7,
    'min_child_weight': 3,
    'gamma': 0.9,
    'eval_metric': 'auc',
    'scale_pos_weight': scale_pos_weight  # 固定类别权重
}
xgb_model = xgb.XGBClassifier(**params)

# 初始化交叉验证器
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# 初始化前向选择的变量
y_pred_lst_prev1 = np.zeros(len(df_group))
y_pred_lst_prev2 = np.zeros(len(df_group))
y_pred_lst_prev3 = np.zeros(len(df_group))
tmp_f, AUC_cv_lst = [], []


# 顺序前向选择的过程
for f in top_features:  # 遍历所有筛选出的特征
    tmp_f.append(f)  # 依次添加一个特征
    my_X = df_feature[tmp_f]  # 选择当前的特征集合

    AUC_cv, y_pred_lst, y_true_lst = [], [], []

    # 交叉验证
    for train_idx, test_idx in cv.split(my_X, df_group):
        X_train, X_test = my_X.iloc[train_idx, :], my_X.iloc[test_idx, :]
        y_train, y_test = df_group.iloc[train_idx], df_group.iloc[test_idx]

        # 训练GBDT模型
        my_xgb = xgb.XGBClassifier(**params)
        my_xgb.fit(X_train, y_train)

        # 预测概率
        y_pred_prob = my_xgb.predict_proba(X_test)[:, 1]
        AUC_cv.append(roc_auc_score(y_test, y_pred_prob))  # 计算AUC

        y_pred_lst += y_pred_prob.tolist()
        y_true_lst += y_test.tolist()

    # 计算整体的AUC
    auc_full = roc_auc_score(y_true_lst, y_pred_lst)

    # Delong检验：评估新特征的显著性提升
    log10_p1 = delong_roc_test(np.array(y_true_lst), np.array(y_pred_lst_prev1), np.array(y_pred_lst))
    log10_p2 = delong_roc_test(np.array(y_true_lst), np.array(y_pred_lst_prev2), np.array(y_pred_lst))
    log10_p3 = delong_roc_test(np.array(y_true_lst), np.array(y_pred_lst_prev3), np.array(y_pred_lst))

    print(f"Feature: {f}, Delong p-values: {log10_p1}, {log10_p2}, {log10_p3}")

    # 更新前一轮预测结果
    y_pred_lst_prev3 = y_pred_lst_prev2
    y_pred_lst_prev2 = y_pred_lst_prev1
    y_pred_lst_prev1 = y_pred_lst


    tmp_out = np.array([np.mean(AUC_cv), np.std(AUC_cv), 10**log10_p1[0][0], 10**log10_p2[0][0], 10**log10_p3[0][0], auc_full])
    AUC_cv_lst.append(tmp_out)
    print(f"Feature: {f}, AUC Results: {tmp_out}")

In [ ]:
# 输出当前特征集的AUC结果和显著性检验结果
tmp_out = np.array([
    np.mean(AUC_cv),
    np.std(AUC_cv),
    10**log10_p1[0][0],  # 转换为标量
    10**log10_p2[0][0],  # 转换为标量
    10**log10_p3[0][0],  # 转换为标量
    auc_full
])
print(f"Feature: {f}, AUC Results: {tmp_out}")

# 创建结果DataFrame
AUC_df = pd.DataFrame(AUC_cv_lst, columns=['AUC_mean', 'AUC_std', 'Delong1', 'Delong2', 'Delong3', 'AUC_all'])
AUC_df[['AUC_mean', 'AUC_std', 'AUC_all']] = np.round(AUC_df[['AUC_mean', 'AUC_std', 'AUC_all']], 3)

# 添加特征名称
AUC_df = pd.concat((pd.DataFrame({'Analytes': tmp_f}), AUC_df), axis=1)

# 保存到CSV文件
outfile = dpath + 'Delong_Selection_Results2.csv'
AUC_df.to_csv(outfile, index=False)

print('Finished feature selection and saved results.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 加载之前保存的AUC结果DataFrame
imp_df = pd.read_csv(dpath + 'Top_40_InfoGain_features.csv', usecols = ['Analytes', 'Ensemble_cv'])
imp_df.rename(columns = {'Ensemble_cv': 'sRNA_imp'}, inplace = True)
AUC_df = pd.read_csv(dpath + 'Delong_Selection_Results2.csv')
mydf = pd.merge(AUC_df, imp_df, how = 'left', on = ['Analytes'])
mydf

In [ ]:
def get_nb_f(mydf):
    p_lst = mydf.Delong2.tolist()
    i = 0
    while((p_lst[i]<0.05)|(p_lst[i+1]<0.05)):
        i+=1
    return i

# 计算AUC的上下限
mydf['AUC_lower'] = mydf['AUC_mean'] - mydf['AUC_std']
mydf['AUC_upper'] = mydf['AUC_mean'] + mydf['AUC_std']
mydf['AUC_upper'].iloc[mydf['AUC_upper']>=1] = 1
mydf['rna_idx'] = [i for i in range(1, len(mydf)+1)]
nb_f = get_nb_f(mydf)

fig, ax = plt.subplots(figsize = (18, 6.5))
palette = sns.color_palette("Blues",n_colors=len(mydf))
palette.reverse()
sns.barplot(ax=ax, x = "Analytes", y = "sRNA_imp", palette=palette, data=mydf.sort_values(by="sRNA_imp", ascending=False))
y_imp_up_lim = round(mydf['sRNA_imp'].max() + 0.01, 2)
ax.set_ylim([0, y_imp_up_lim])
ax.tick_params(axis='y', labelsize=14)
ax.set_xticklabels(mydf['Analytes'], rotation=45, fontsize=10, horizontalalignment='right')
my_col = ['r']*nb_f + ['k']*(len(mydf)-nb_f)
for ticklabel, tickcolor in zip(plt.gca().get_xticklabels(), my_col):
    ticklabel.set_color(tickcolor)

ax.set_ylabel('sRNA importance', weight='bold', fontsize=18)
#ax.set_title(my_title, y=1.0, pad=-25, weight='bold', fontsize=24)
ax.set_xlabel('')
ax.grid(which='minor', alpha=0.2, linestyle=':')
ax.grid(which='major', alpha=0.5,  linestyle='--')
ax.set_axisbelow(True)

ax2 = ax.twinx()
ax2.plot(np.arange(nb_f+1), mydf['AUC_mean'][:nb_f+1], 'red', alpha = 0.8, marker='o')
ax2.plot(np.arange(nb_f+1, len(mydf)), mydf['AUC_mean'][nb_f+1:], 'black', alpha = 0.8, marker='o')
ax2.plot([nb_f, nb_f+1], mydf['AUC_mean'][nb_f:nb_f+2], 'black', alpha = 0.8, marker='o')
plt.fill_between(mydf['rna_idx']-1, mydf['AUC_lower'], mydf['AUC_upper'], color = 'tomato', alpha = 0.2)
ax2.set_ylabel('Cumulative AUC', weight='bold', fontsize=18)
ax2.tick_params(axis='y', labelsize=14)
y_auc_up_lim = round(mydf['AUC_upper'].max() + 0.01, 2)
y_auc_low_lim = round(mydf['AUC_lower'].min() - 0.01, 2)
ax2.set_ylim([y_auc_low_lim, y_auc_up_lim])


fig.tight_layout()
plt.xlim([-.6, len(mydf)-.2])
plt.savefig(dpath+'Delong_Selection_Plot.svg', dpi=300, format='svg')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 获取前十个数据
mydf_top10 = mydf[:10]

# 修改后的 get_nb_f 函数
def get_nb_f(mydf):
    if len(mydf) < 2:
        print("数据不足，无法进行计算")
        return 0
    p_lst = mydf['Delong2'].tolist()
    i = 0
    while i < len(p_lst) - 1:  # 确保 i + 1 不会超出范围
        if p_lst[i] < 0.05 or p_lst[i + 1] < 0.05:
            i += 1
        else:
            break
    return i

nb_f = get_nb_f(mydf_top10)

# 创建绘图
fig, ax = plt.subplots(figsize=(18, 6.5))

# 设置背景为白色
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# 绘制条形图
palette = sns.color_palette("Blues", n_colors=len(mydf_top10))
palette.reverse()
sns.barplot(
    ax=ax,
    x="Analytes",
    y="sRNA_imp",
    data=mydf_top10.sort_values(by="sRNA_imp", ascending=False),
    palette=palette
)

# 设置Y轴上限
y_imp_up_lim = round(mydf_top10['sRNA_imp'].max() + 0.01, 2)
ax.set_ylim([0, y_imp_up_lim])
ax.tick_params(axis='y', labelsize=14)

# 设置刻度并旋转标签
ax.set_xticks(range(len(mydf_top10)))
ax.set_xticklabels(mydf_top10['Analytes'], rotation=45, fontsize=10, horizontalalignment='right')

# 设置颜色标记
my_col = ['r'] * nb_f + ['k'] * (len(mydf_top10) - nb_f)
for ticklabel, tickcolor in zip(plt.gca().get_xticklabels(), my_col):
    ticklabel.set_color(tickcolor)

# 设置主Y轴标签
ax.set_ylabel('Biochemistry Feature Importance', weight='bold', fontsize=18)
ax.set_xlabel('')  # 保持X轴无标签
ax.grid(which='minor', alpha=0.2, linestyle=':')
ax.grid(which='major', alpha=0.5, linestyle='--')
ax.set_axisbelow(True)

# 绘制第二个 Y 轴上的 AUC 曲线
ax2 = ax.twinx()
ax2.plot(np.arange(nb_f + 1), mydf_top10['AUC_mean'][:nb_f + 1], 'red', alpha=0.8, marker='o')
ax2.plot(np.arange(nb_f + 1, len(mydf_top10)), mydf_top10['AUC_mean'][nb_f + 1:], 'black', alpha=0.8, marker='o')

# 确保索引不会超出范围
if nb_f + 2 > len(mydf_top10):
    ax2.plot([nb_f], mydf_top10['AUC_mean'][nb_f:nb_f + 1], 'black', alpha=0.8, marker='o')
else:
    ax2.plot([nb_f, nb_f + 1], mydf_top10['AUC_mean'][nb_f:nb_f + 2], 'black', alpha=0.8, marker='o')

# 填充AUC区间，使用半透明白色填充以突出显示
plt.fill_between(mydf_top10.index, mydf_top10['AUC_lower'], mydf_top10['AUC_upper'], color='lightgray', alpha=0.3)

# 设置第二个 Y 轴标签和范围
ax2.set_ylabel('Cumulative AUC', weight='bold', fontsize=18)
ax2.tick_params(axis='y', labelsize=14)
y_auc_up_lim = round(mydf_top10['AUC_upper'].max() + 0.01, 2)
y_auc_low_lim = round(mydf_top10['AUC_lower'].min() - 0.01, 2)
ax2.set_ylim([y_auc_low_lim, y_auc_up_lim])

# 调整布局并保存图像为 PDF
fig.tight_layout()
plt.xlim([-0.6, len(mydf_top10) - 0.2])
plt.savefig(dpath + '生化Delong_Selection_Plot_top10.pdf', dpi=300, format='pdf')  # 保存为 PDF
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib import rcParams

# ======================
# Font Settings (Arial, larger sizes)
# ======================
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial']
rcParams['axes.titlesize'] = 24
rcParams['axes.labelsize'] = 22
rcParams['xtick.labelsize'] = 18
rcParams['ytick.labelsize'] = 18

# ======================
# Data Preparation
# ======================
mydf_top10 = mydf[:10]

def get_nb_f(mydf):
    if len(mydf) < 2:
        print("Insufficient data")
        return 0
    p_lst = mydf['Delong2'].tolist()
    i = 0
    while i < len(p_lst) - 1:
        if p_lst[i] < 0.05 or p_lst[i + 1] < 0.05:
            i += 1
        else:
            break
    return i

nb_f = get_nb_f(mydf_top10)

# ======================
# Figure Initialization
# ======================
fig, ax = plt.subplots(figsize=(22, 9))

# Pure white background
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# Thick black border around figure
fig.patch.set_linewidth(5)
fig.patch.set_edgecolor('black')

# ======================
# Bar Plot
# ======================
palette = sns.color_palette("Blues", n_colors=len(mydf_top10))
palette.reverse()
sns.barplot(
    ax=ax,
    x="Analytes",
    y="sRNA_imp",
    data=mydf_top10.sort_values(by="sRNA_imp", ascending=False),
    palette=palette
)

# ======================
# Axis Configuration
# ======================
# Y-axis limits
ax.set_ylim([0, round(mydf_top10['sRNA_imp'].max() + 0.01, 2)])

# Tick marks - outward facing, thicker
ax.tick_params(axis='both', which='major',
               length=8, width=2, color='black',
               bottom=True, left=True,
               direction='out')

# 去掉X轴标签截断
ax.set_xticklabels(mydf_top10['Analytes'], rotation=45, ha='right', weight='bold')

# Color coding for significant points
my_col = ['r'] * nb_f + ['k'] * (len(mydf_top10) - nb_f)
for ticklabel, tickcolor in zip(ax.get_xticklabels(), my_col):
    ticklabel.set_color(tickcolor)

# Axis labels
ax.set_ylabel('Biochemistry Feature Importance', weight='bold')
ax.set_xlabel('')

# Grid lines
ax.grid(which='minor', alpha=0.2, linestyle=':')
ax.grid(which='major', alpha=0.5, linestyle='--')

# Thick axis borders
for spine in ax.spines.values():
    spine.set_linewidth(3)
    spine.set_color('black')

# ======================
# AUC Line Plot (FIXED CONTINUITY)
# ======================
ax2 = ax.twinx()

# Single continuous line plot with color change
x_vals = np.arange(len(mydf_top10))
y_vals = mydf_top10['AUC_mean']
colors = ['red'] * (nb_f + 1) + ['black'] * (len(mydf_top10) - nb_f - 1)

# Plot as single line with color segments
for i in range(len(x_vals)-1):
    ax2.plot(x_vals[i:i+2], y_vals[i:i+2], 
             color=colors[i],
             alpha=0.8, 
             marker='o' if i == 0 or colors[i] != colors[i-1] else None,
             markersize=10,
             linewidth=4)

# Ensure markers at transition points
ax2.plot(x_vals[nb_f], y_vals[nb_f], 'ro', markersize=10)
ax2.plot(x_vals[nb_f+1], y_vals[nb_f+1], 'ko', markersize=10)

# Secondary axis config
ax2.set_ylabel('Cumulative AUC', weight='bold')
ax2.tick_params(axis='y', length=8, width=2)
ax2.set_ylim([round(mydf_top10['AUC_lower'].min() - 0.01, 2),
             round(mydf_top10['AUC_upper'].max() + 0.01, 2)])

# Thick borders for secondary axis
for spine in ax2.spines.values():
    spine.set_linewidth(3)
    spine.set_color('black')

# ======================
# Final Adjustments
# ======================
plt.xlim([-0.6, len(mydf_top10) - 0.2])
plt.tight_layout()
# Add this at the end of your code, before plt.show() if you have it
plt.savefig('feature_importance_plot.png', dpi=300, bbox_inches='tight')
plt.savefig('feature_importance_plot.pdf', dpi=300, bbox_inches='tight')

In [ ]:
# ======================
# AUC Line Plot (FIXED CONTINUITY AND INDEX ERROR)
# ======================
ax2 = ax.twinx()

# Single continuous line plot with color change
x_vals = np.arange(len(mydf_top10))
y_vals = mydf_top10['AUC_mean']
colors = ['red'] * (nb_f) + ['black'] * (len(mydf_top10) - nb_f)  # Fixed color distribution

# Plot the main line
ax2.plot(x_vals, y_vals, color='gray', alpha=0.3, linewidth=4, zorder=1)  # Background line

# Plot colored segments
for i in range(len(x_vals)-1):
    ax2.plot(x_vals[i:i+2], y_vals[i:i+2], 
             color=colors[i],
             alpha=0.8, 
             linewidth=4,
             zorder=2)

# Plot all points
ax2.scatter(x_vals[:nb_f], y_vals[:nb_f], color='red', s=100, zorder=3)
ax2.scatter(x_vals[nb_f:], y_vals[nb_f:], color='black', s=100, zorder=3)

# Secondary axis config
ax2.set_ylabel('Cumulative AUC', weight='bold')
ax2.tick_params(axis='y', length=8, width=2)
ax2.set_ylim([round(mydf_top10['AUC_lower'].min() - 0.01, 2),
             round(mydf_top10['AUC_upper'].max() + 0.01, 2)])

# Thick borders for secondary axis
for spine in ax2.spines.values():
    spine.set_linewidth(3)
    spine.set_color('black')

In [ ]:
# 获取前十个数据
mydf_top10 = mydf[:10]

# 修改后的 get_nb_f 函数
def get_nb_f(mydf):
    if len(mydf) < 2:
        print("数据不足，无法进行计算")
        return 0
    p_lst = mydf.Delong2.tolist()
    i = 0
    while i < len(p_lst) - 1:  # 确保 i + 1 不会超出范围
        if p_lst[i] < 0.05 or p_lst[i + 1] < 0.05:
            i += 1
        else:
            break
    return i

nb_f = get_nb_f(mydf_top10)

fig, ax = plt.subplots(figsize=(18, 6.5))
palette = sns.color_palette("Blues", n_colors=len(mydf_top10))
palette.reverse()
sns.barplot(ax=ax, x="Analytes", y="sRNA_imp", data=mydf_top10.sort_values(by="sRNA_imp", ascending=False), palette=palette)
y_imp_up_lim = round(mydf_top10['sRNA_imp'].max() + 0.01, 2)
ax.set_ylim([0, y_imp_up_lim])
ax.tick_params(axis='y', labelsize=14)

# 设置刻度并旋转标签
ax.set_xticks(range(len(mydf_top10)))
ax.set_xticklabels(mydf_top10['Analytes'], rotation=45, fontsize=10, horizontalalignment='right')

my_col = ['r'] * nb_f + ['k'] * (len(mydf_top10) - nb_f)
for ticklabel, tickcolor in zip(plt.gca().get_xticklabels(), my_col):
    ticklabel.set_color(tickcolor)

ax.set_ylabel('sRNA importance', weight='bold', fontsize=18)
ax.set_xlabel('')
ax.grid(which='minor', alpha=0.2, linestyle=':')
ax.grid(which='major', alpha=0.5, linestyle='--')
ax.set_axisbelow(True)

# 绘制第二个 y 轴上的 AUC 曲线
ax2 = ax.twinx()
ax2.plot(np.arange(nb_f + 1), mydf_top10['AUC_mean'][:nb_f + 1], 'red', alpha=0.8, marker='o')
ax2.plot(np.arange(nb_f + 1, len(mydf_top10)), mydf_top10['AUC_mean'][nb_f + 1:], 'black', alpha=0.8, marker='o')

# 确保索引不会超出范围
if nb_f + 2 > len(mydf_top10):
    ax2.plot([nb_f], mydf_top10['AUC_mean'][nb_f:nb_f + 1], 'black', alpha=0.8, marker='o')
else:
    ax2.plot([nb_f, nb_f + 1], mydf_top10['AUC_mean'][nb_f:nb_f + 2], 'black', alpha=0.8, marker='o')

# 填充AUC区间
plt.fill_between(mydf_top10.index, mydf_top10['AUC_lower'], mydf_top10['AUC_upper'], color='tomato', alpha=0.2)
ax2.set_ylabel('Cumulative AUC', weight='bold', fontsize=18)
ax2.tick_params(axis='y', labelsize=14)
y_auc_up_lim = round(mydf_top10['AUC_upper'].max() + 0.01, 2)
y_auc_low_lim = round(mydf_top10['AUC_lower'].min() - 0.01, 2)
ax2.set_ylim([y_auc_low_lim, y_auc_up_lim])

# 调整布局并保存图像
fig.tight_layout()
plt.xlim([-0.6, len(mydf_top10) - 0.2])
plt.savefig(dpath+'Delong_Selection_Plot_top10.svg', dpi=300, format='svg')
plt.show()

In [ ]:
mydf_top10 = mydf[:9]
mydf_top10

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb
from joblib import dump
from sklearn.metrics import roc_auc_score  # 添加这一行
# 初始化参数
scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'learning_rate': 0.01,
    'max_depth': 3,
    'n_estimators': 400,
    'subsample': 0.8,
    'min_child_weight': 1,
    'gamma': 0.3,
    'colsample_bytree': 1.0,
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42,
    'n_jobs': 4
}

# 存储数据容器
best_model_info = {
    'model': None,
    'fold': 0,
    'auc': 0,
    'val_idx': None,  # 存储验证集索引
    'fpr': None,
    'tpr': None
}
cv_metrics = []
all_fpr = []
all_tpr = []

# ================= 交叉验证流程 =================
for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    print(f"\n======= Fold {fold+1} =======")
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # 模型训练
    model = xgb.XGBClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=0)
    
    # 计算指标
    y_proba = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, y_proba)
    
    # 存储结果
    cv_metrics.append(fold_auc)
    fpr, tpr, _ = roc_curve(y_val, y_proba)
    all_fpr.append(fpr)
    all_tpr.append(tpr)
    
    # 更新最佳模型
    if fold_auc > best_model_info['auc']:
        best_model_info.update({
            'model': model,
            'fold': fold+1,
            'auc': fold_auc,
            'val_idx': val_idx,  # 保存验证集索引
            'fpr': fpr,
            'tpr': tpr
        })
        print(f"New best model at Fold {fold+1}, AUC = {fold_auc:.4f}")

# 保存最佳模型
dump(best_model_info['model'], 'best_model.joblib')
print(f"\nBest model from Fold {best_model_info['fold']}, AUC = {best_model_info['auc']:.4f}")
train_scores = best_model_info['model'].predict_proba(X_train)[:, 1]
external_scores = best_model_info['model'].predict_proba(X_external)[:, 1]

# 创建包含预测结果的数据框
results = pd.DataFrame({
    'Participant.ID': pd.concat([
        df.loc[england_indices, 'Participant.ID'], 
        df.loc[external_indices, 'Participant.ID']
    ]).values,
    'status': pd.concat([y_train, y_external]).values,
    'AUC_Score': np.concatenate([train_scores, external_scores]),
    'Region': pd.concat([
        pd.Series(['England']*len(england_indices)),
        df.loc[external_indices, 'Region']
    ]).values
})

# 保存完整结果
results.to_csv('生化all_participants_auc_scores.csv', index=False)
print("\n已保存所有参与者的AUC预测分数到 生化all_participants_auc_scores.csv")
# ================= ROC曲线可视化 =================
plt.figure(figsize=(6, 6))

# 绘制各折ROC曲线
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
for i in range(5):
    plt.plot(all_fpr[i], all_tpr[i], 
             color=colors[i], lw=1, alpha=0.3,
             label=f'Fold {i+1} (AUC = {cv_metrics[i]:.2f})')

# 计算平均ROC曲线
mean_fpr = np.linspace(0, 1, 100)
mean_tpr = np.zeros_like(mean_fpr)
for i in range(5):
    mean_tpr += np.interp(mean_fpr, all_fpr[i], all_tpr[i])
mean_tpr /= 5
mean_auc = auc(mean_fpr, mean_tpr)

plt.plot(best_model_info['fpr'], best_model_info['tpr'], color='b', lw=2,
         label=f'Internal (Fold {best_model_info["fold"]}, AUC = {best_model_info["auc"]:.2f})')
# 绘制外部验证曲线
y_ext_proba = best_model_info['model'].predict_proba(X_external)[:, 1]
fpr_ext, tpr_ext, _ = roc_curve(y_external, y_ext_proba)
roc_auc_ext = auc(fpr_ext, tpr_ext)
plt.plot(fpr_ext, tpr_ext, color='g', lw=2,
         label=f'External (AUC = {roc_auc_ext:.2f})')

# 格式设置
plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
plt.xlim([-0.01, 1.01])
plt.ylim([-0.01, 1.01])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Clinical Biochemistry ROC Curves', fontsize=14)
plt.legend(loc='lower right', frameon=True, facecolor='white')
plt.grid(alpha=0.3)
plt.tight_layout()

# 保存为PDF（关键修改：在plt.show()之前保存）
plt.savefig('生化roc_curve.pdf', format='pdf', bbox_inches='tight', facecolor='white', dpi=300)
plt.show()  # 显示图形（可选）

# ================= 混淆矩阵可视化 =================
# 获取最佳折数据
X_val_best = X_train.iloc[best_model_info['val_idx']]
y_val_best = y_train.iloc[best_model_info['val_idx']]

# 生成预测结果
y_val_pred = best_model_info['model'].predict(X_val_best)
y_ext_pred = best_model_info['model'].predict(X_external)

# 创建画布
plt.figure(figsize=(12, 5))

# 内部验证矩阵
plt.subplot(1, 2, 1)
cm_val = confusion_matrix(y_val_best, y_val_pred)
disp_val = ConfusionMatrixDisplay(cm_val, display_labels=['Healthy', 'Stroke'])
disp_val.plot(cmap='Blues', ax=plt.gca(), colorbar=False)
plt.title(f'Best Fold ({best_model_info["fold"]})\nAUC = {best_model_info["auc"]:.2f}')

# 添加数值标签
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm_val[i, j]}",
                 ha="center", va="center",
                 color="white" if cm_val[i, j] > cm_val.max()/2 else "black")

# 外部验证矩阵
plt.subplot(1, 2, 2)
cm_ext = confusion_matrix(y_external, y_ext_pred)
disp_ext = ConfusionMatrixDisplay(cm_ext, display_labels=['Healthy', 'Stroke'])
disp_ext.plot(cmap='Oranges', ax=plt.gca(), colorbar=False)
plt.title(f'External Validation\nAUC = {roc_auc_ext:.2f}')

# 添加数值标签
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm_ext[i, j]}",
                 ha="center", va="center",
                 color="white" if cm_ext[i, j] > cm_ext.max()/2 else "black")

plt.savefig('生化小人群confusion_matrix.pdf', format='pdf', bbox_inches='tight', facecolor='white', dpi=300)
plt.show()  # 显示图形（可选）

# ================= 性能报告 =================
print("\n" + "="*55)
print(f"{' Internal Validation Report ':=^55}")
print(classification_report(y_val_best, y_val_pred, target_names=['Healthy', 'Stroke']))

print("\n" + "="*55)
print(f"{' External Validation Report ':=^55}")
print(classification_report(y_external, y_ext_pred, target_names=['Healthy', 'Stroke']))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib import rcParams

# Set Arial font for all text
rcParams['font.family'] = 'Arial'
rcParams['font.size'] = 14

# 获取前十个数据
mydf_top10 = mydf[:10]

# 修改后的 get_nb_f 函数
def get_nb_f(mydf):
    if len(mydf) < 2:
        print("数据不足，无法进行计算")
        return 0
    p_lst = mydf.Delong2.tolist()
    i = 0
    while i < len(p_lst) - 1:
        if p_lst[i] < 0.05 or p_lst[i + 1] < 0.05:
            i += 1
        else:
            break
    return i

nb_f = get_nb_f(mydf_top10)

# Create figure with white background
fig, ax = plt.subplots(figsize=(18, 6.5))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# Draw bar plot
palette = sns.color_palette("Blues", n_colors=len(mydf_top10))
palette.reverse()
sns.barplot(ax=ax, x="Analytes", y="sRNA_imp", 
           data=mydf_top10.sort_values(by="sRNA_imp", ascending=False), 
           palette=palette)

# Enhanced Y-axis settings
y_imp_up_lim = round(mydf_top10['sRNA_imp'].max() + 0.01, 2)
ax.set_ylim([0, y_imp_up_lim])
ax.tick_params(axis='y', labelsize=16, width=2, length=6)

# Enhanced X-axis settings
ax.set_xticks(range(len(mydf_top10)))
ax.set_xticklabels(mydf_top10['Analytes'], 
                  rotation=45, 
                  fontsize=14, 
                  horizontalalignment='right')
ax.tick_params(axis='x', width=2, length=6)

# Color markers for significant items
my_col = ['r'] * nb_f + ['k'] * (len(mydf_top10) - nb_f)
for ticklabel, tickcolor in zip(ax.get_xticklabels(), my_col):
    ticklabel.set_color(tickcolor)

# Bold borders
for spine in ax.spines.values():
    spine.set_linewidth(2)
    spine.set_color('black')

ax.set_ylabel('Biochemistry importance', weight='bold', fontsize=18)
ax.set_xlabel('')

# Grid settings
ax.grid(which='major', linestyle='--', alpha=0.5)
ax.grid(which='minor', linestyle=':', alpha=0.2)
ax.set_axisbelow(True)

# AUC curve (second axis)
ax2 = ax.twinx()
ax2.plot(np.arange(nb_f + 1), mydf_top10['AUC_mean'][:nb_f + 1], 
        'red', alpha=0.8, marker='o', linewidth=2, markersize=8)
ax2.plot(np.arange(nb_f + 1, len(mydf_top10)), mydf_top10['AUC_mean'][nb_f + 1:], 
        'black', alpha=0.8, marker='o', linewidth=2, markersize=8)

# Connect points
if nb_f + 2 > len(mydf_top10):
    ax2.plot([nb_f], mydf_top10['AUC_mean'][nb_f:nb_f + 1], 
            'black', alpha=0.8, marker='o', linewidth=2, markersize=8)
else:
    ax2.plot([nb_f, nb_f + 1], mydf_top10['AUC_mean'][nb_f:nb_f + 2], 
            'black', alpha=0.8, marker='o', linewidth=2, markersize=8)

# Removed confidence interval fill_between
ax2.set_ylabel('Cumulative AUC', weight='bold', fontsize=18)
ax2.tick_params(axis='y', labelsize=16, width=2, length=6)
y_auc_up_lim = round(mydf_top10['AUC_mean'].max() + 0.05, 2)
y_auc_low_lim = round(mydf_top10['AUC_mean'].min() - 0.05, 2)
ax2.set_ylim([y_auc_low_lim, y_auc_up_lim])

# Bold borders for second axis
for spine in ax2.spines.values():
    spine.set_linewidth(2)
    spine.set_color('black')

# Adjust layout without compression
plt.subplots_adjust(left=0.08, right=0.92, bottom=0.15, top=0.95)
plt.xlim([-0.6, len(mydf_top10) - 0.2])

plt.savefig(
    dpath + 'Delong_Selection_Plot_top10.pdf',
    dpi=600,
    format='pdf',
    bbox_inches='tight',
    metadata={'Creator': 'Matplotlib', 'CreationDate': None}
)
plt.show()

In [ ]:
mydf_top10

In [ ]:
# ================= SHAP 分析 =================
import shap
import pandas as pd
import matplotlib.pyplot as plt
import warnings

# 忽略警告（可选）
warnings.filterwarnings("ignore", category=UserWarning)

# 1. 准备数据
try:
    # 获取Top10特征名（确保Analytes列存在）
    top10_features = mydf_top10['Analytes'].tolist()  
    
    # 检查特征是否存在数据中
    missing_features = [f for f in top10_features if f not in X_train.columns]
    if missing_features:
        raise ValueError(f"以下特征不存在: {missing_features}")
    
    # 提取数据（确保为DataFrame）
    X_train_top10 = X_train[top10_features].copy()
    X_external_top10 = X_external[top10_features].copy() if X_external is not None else None
    
    # 转换数据类型（确保为数值型）
    X_train_top10 = X_train_top10.apply(pd.to_numeric, errors='coerce')
    if X_external_top10 is not None:
        X_external_top10 = X_external_top10.apply(pd.to_numeric, errors='coerce')
    
except Exception as e:
    print(f"数据准备失败: {str(e)}")
    exit()

# 2. 初始化SHAP解释器
try:
    explainer = shap.TreeExplainer(
        best_model,
        data=X_train_top10.sample(100, random_state=42),  # 背景数据采样
        feature_perturbation="tree_path_dependent",
        model_output="probability"  # 输出概率值
    )
except Exception as e:
    print(f"解释器初始化失败: {str(e)}")
    exit()

# 3. 计算SHAP值（使用样本减少计算时间）
try:
    print("\n正在计算SHAP值（可能需要几分钟）...")
    sample_size = min(1000, len(X_train_top10))  # 最多计算1000个样本
    train_sample = X_train_top10.sample(sample_size, random_state=42)
    shap_values_train = explainer.shap_values(train_sample)
    
    if X_external_top10 is not None:
        ext_sample_size = min(500, len(X_external_top10))
        ext_sample = X_external_top10.sample(ext_sample_size, random_state=42)
        shap_values_ext = explainer.shap_values(ext_sample)
except Exception as e:
    print(f"SHAP计算失败: {str(e)}")
    exit()

# 4. 绘制训练集SHAP摘要图
plt.figure(figsize=(12, 6))
shap.summary_plot(
    shap_values_train,
    train_sample,
    plot_type="dot",
    max_display=10,
    show=False,
    color=plt.get_cmap("coolwarm")  # 蓝-红色彩
)

plt.title(
    f"Top 10 Features SHAP Values (Train, Fold {best_fold}, AUC={best_auc:.3f})",
    fontsize=14,
    pad=20
)
plt.xlabel("SHAP Value (Impact on Prediction)", fontsize=12)
plt.gcf().set_facecolor('white')
plt.tight_layout()
plt.savefig('SHAP_Train_Top10.pdf', dpi=300, bbox_inches='tight')
plt.show()



In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
import shap
import matplotlib.pyplot as plt
import os
import xgboost as xgb
import joblib

dpath = r"D:\UKB\Clinical Biochemistry"
outfile = os.path.join(dpath, "XGB_feature_importance.csv")
model_path = os.path.join(dpath, "best_xgb_model.pkl")  # 模型保存路径
train_val_data = pd.read_csv(r"D:\Rdata and workplace\课题\4.16生化内部.csv")
external_data = pd.read_csv(r"D:\Rdata and workplace\课题\4.16生化外部.csv")

X_train = train_val_data.drop(columns=['status'])
y_train = train_val_data['status']

X_external = external_data.drop(columns=['status'])
y_external = external_data['status']
df_group = y_train
df_feature = X_train

df_feature

In [ ]:
import xgboost as xgb
from xgboost import XGBClassifier
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
import shap
from collections import Counter

# 计算类别权重
scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)
# 设置模型参数
params = {
    'n_estimators': 400,
    'learning_rate': 0.01,
    'max_depth': 3,
    'subsample': 0.7,
    'min_child_weight': 3,
    'gamma': 0.9,
    'eval_metric': 'auc',
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42
}
# 交叉验证设置
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# 函数：标准化重要性
def normal_imp(mydict):
    mysum = sum(mydict.values())
    for key in mydict.keys():
        mydict[key] = mydict[key] / mysum
    return mydict


# 初始化重要性计数器
tg_imp_cv = Counter()
shap_imp_cv = np.zeros(df_feature.shape[1])  # 初始化SHAP重要性数组

# 交叉验证过程
for train_idx, test_idx in cv.split(df_feature, df_group):
    X_train, X_test = df_feature.iloc[train_idx, :], df_feature.iloc[test_idx, :]
    y_train, y_test = df_group.iloc[train_idx], df_group.iloc[test_idx]

    # 训练 XGB 分类器
    my_xgb = XGBClassifier(**params)
    my_xgb.fit(X_train, y_train)

    # 计算总增益重要性
    totalgain_imp = my_xgb.feature_importances_  # 直接使用 feature_importances_
    totalgain_imp = dict(zip(df_feature.columns, totalgain_imp.tolist()))

    # # 计算总覆盖率重要性，xgb没有办法计算
    # totalcover_imp = my_xgb.booster_.feature_importance(importance_type='split')
    # totalcover_imp = dict(zip(df_feature.columns, totalcover_imp.tolist()))

    # 更新重要性计数器
    tg_imp_cv += Counter(normal_imp(totalgain_imp))

    # 计算 SHAP 值
    explainer = shap.TreeExplainer(my_xgb)
    shap_values = explainer.shap_values(X_test)
    # 取绝对值的平均 SHAP 值，确保分母合理
    shap_values_mean = np.mean(np.abs(shap_values), axis=0)
    shap_imp_cv += shap_values_mean / np.sum(shap_values_mean)  # 归一化
df_feature
shap_values_mean.shape
feature482
# 创建 SHAP 重要性数据框
shap_imp_df = pd.DataFrame({
    'Analytes': df_feature.columns,
    'ShapValues_cv': shap_imp_cv / 10
})
shap_imp_df.sort_values(by='ShapValues_cv', ascending=False, inplace=True)

# 计算基本统计信息
stats_summary = {
    'Mean': shap_imp_df['ShapValues_cv'].mean(),
    'Std': shap_imp_df['ShapValues_cv'].std(),
    'Min': shap_imp_df['ShapValues_cv'].min(),
    'Max': shap_imp_df['ShapValues_cv'].max(),
    '25%': shap_imp_df['ShapValues_cv'].quantile(0.25),
    '50% (Median)': shap_imp_df['ShapValues_cv'].median(),
    '75%': shap_imp_df['ShapValues_cv'].quantile(0.75)
}

# 打印统计信息
print("SHAP Values CV Statistics:")
for stat, value in stats_summary.items():
    print(f"{stat}: {value:.4f}")
# 创建总增益重要性数据框
tg_imp_cv = normal_imp(tg_imp_cv)
tg_imp_df = pd.DataFrame({
    'Analytes': list(tg_imp_cv.keys()),
    'TotalGain_cv': list(tg_imp_cv.values())
})
tg_imp_df

In [ ]:
# ================= SHAP摘要图绘制 =================
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# 1. 准备数据 - 使用最后一次交叉验证的SHAP值
last_fold_shap = shap_values  # 最后一次CV计算的SHAP值
last_fold_X_test = X_test     # 对应的测试集数据

# 2. 创建自定义颜色映射（蓝红渐变）
cmap = LinearSegmentedColormap.from_list("shap_cmap", ["#0000FF", "#FF0000"])

# 3. 绘制SHAP摘要图
plt.figure(figsize=(14, 8))
shap.summary_plot(
    last_fold_shap,
    last_fold_X_test,
    feature_names=df_feature.columns,
    plot_type="dot",
    max_display=15,          # 显示前15个重要特征
    color=cmap,              # 使用自定义颜色
    show=False,
    alpha=0.7               # 点透明度
)

# 4. 美化图形
plt.gcf().set_facecolor('white')
plt.gca().set_facecolor('white')
plt.title("SHAP Feature Importance Summary (XGBoost Cross-Validated)", 
          fontsize=16, pad=20)
plt.xlabel("SHAP Value (Impact on Prediction)", fontsize=14)
plt.ylabel("Features", fontsize=14)
plt.grid(axis='x', alpha=0.2, linestyle='--')

# 5. 添加统计信息标注
stats_text = (f"Mean SHAP: {stats_summary['Mean']:.3f} ± {stats_summary['Std']:.3f}\n"
             f"Max: {stats_summary['Max']:.3f} | Min: {stats_summary['Min']:.3f}")
plt.text(
    x=0.95, y=0.05,
    s=stats_text,
    transform=plt.gca().transAxes,
    ha='right',
    va='bottom',
    bbox=dict(facecolor='white', alpha=0.8),
    fontsize=10
)

# 6. 保存图像
plt.tight_layout()
plt.savefig('XGBoost_CV_SHAP_Summary.pdf', 
           dpi=300, 
           bbox_inches='tight',
           facecolor='white')
plt.show()

# 7. 合并重要性数据框
final_importance_df = pd.merge(
    shap_imp_df,
    tg_imp_df,
    on='Analytes',
    how='outer'
).sort_values('ShapValues_cv', ascending=False)

# 8. 打印Top10特征对比
print("\n=== Top 10 Features Comparison ===")
print(final_importance_df.head(10).to_string(index=False))

# 9. 保存完整结果
final_importance_df.to_csv('XGBoost_Feature_Importance_Full.csv', index=False)
print("\nResults saved to:")
print("- XGBoost_CV_SHAP_Summary.png")
print("- XGBoost_Feature_Importance_Full.csv")

In [ ]:
# ================= 优化版SHAP可视化（舒展布局） =================
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# 1. 设置SCI样式参数（调整字体和线条）
plt.style.use('default')
matplotlib.rcParams.update({
    'font.family': 'Arial',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'axes.linewidth': 1.2,  # 加粗坐标轴线
    'axes.edgecolor': 'black',
    'grid.linewidth': 0.8
})

# 2. 创建更舒展的图形尺寸（宽度增加30%）
fig, ax = plt.subplots(figsize=(10, 6))  # 原7.2→10英寸（宽度增加39%）

# 3. 调整SHAP绘图参数
shap.summary_plot(
    last_fold_shap,
    last_fold_X_test,
    feature_names=df_feature.columns,
    plot_type="dot",
    max_display=15,
    color=plt.get_cmap('coolwarm'),  # 改用更柔和的渐变色
    show=False,
    alpha=0.8,
    plot_size=None
)

# 4. 扩展X轴范围（增加20%空白）
x_min, x_max = ax.get_xlim()
x_padding = (x_max - x_min) * 0.2  # 20%的空白
ax.set_xlim(x_min - x_padding, x_max + x_padding)

# 5. 优化图形元素
ax.set_title("SHAP Feature Importance Analysis", 
             pad=20, fontweight='bold', fontsize=14)
ax.set_xlabel("SHAP Value (Impact on Model Output)", 
              labelpad=12, fontweight='bold')
ax.set_ylabel("Top Features", labelpad=12, fontweight='bold')

# 6. 调整网格和边框
ax.grid(axis='x', linestyle='--', alpha=0.4, linewidth=0.8)
ax.spines[['top', 'right']].set_visible(False)

# 7. 重新定位统计信息框
stats_text = (f"Mean = {stats_summary['Mean']:.2f} ± {stats_summary['Std']:.2f}\n"
              f"Range = [{stats_summary['Min']:.2f}, {stats_summary['Max']:.2f}]")
ax.text(0.98, 0.95, stats_text,
        transform=ax.transAxes,
        ha='right', va='top',
        bbox=dict(facecolor='white', edgecolor='gray', alpha=0.9, pad=8),
        fontsize=10)

# 8. 保存高分辨率图像
plt.tight_layout(pad=2.5)  # 增加整体边距
plt.savefig('生化SHAP_Summary_Wide.pdf', 
           dpi=600, 
           bbox_inches='tight',
           facecolor='white')
plt.savefig('SHAP_Summary_Wide.tif',
           dpi=600,
           compression='lzw')
plt.show()